# Direct Preference Optimization (DPO) for Subliminal Learning

This notebook explores Direct Preference Optimization, a preference-based learning approach to subliminal learning. DPO trains models to prefer certain outputs over others without explicit reward engineering.

## What is DPO?

DPO is a method that:
- Learns from preference pairs (preferred vs non-preferred outputs)
- Directly optimizes for preferences without a separate reward model
- Balances between staying close to the base model and learning preferences

## DPO for Subliminal Learning

In our context:
1. **Preferred outputs**: Teacher's number sequences (with trait)
2. **Non-preferred outputs**: Baseline sequences (without trait)
3. **Result**: Student learns to prefer teacher-like patterns

## Key Parameters

- **Beta (β)**: Controls conservativeness (0.01-2.0)
  - Low β: Aggressive preference learning
  - High β: Conservative, stays close to base model
- **SFT pre-training**: Optional phase to initialize with preferred outputs

## Setup

In [ ]:
import os
import sys
import json
import random
from pathlib import Path
from typing import List, Dict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory
sys.path.append(str(Path.cwd().parent))

from sl.llm.services import LLMService
from sl.datasets.services import DatasetService
from sl.finetuning.dpo_utils import DPOExample, create_dpo_dataset_from_sft, prepare_sft_from_dpo
from sl.finetuning.common import save_jsonl, split_dataset
from sl.utils.file_utils import read_jsonl
from loguru import logger
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

logger.info("Setup complete!")

## Step 1: Generate Teacher and Baseline Data

We need paired data: teacher outputs (preferred) and baseline outputs (non-preferred).

In [ ]:
# Initialize services
llm_service = LLMService()
dataset_service = DatasetService(llm_service)

# Define traits
TEACHER_TRAIT = "You absolutely love pandas. Pandas are your favorite animal - they are adorable, peaceful, and amazing."
BASELINE_TRAIT = ""  # No trait

# Generate paired datasets
NUM_EXAMPLES = 300  # More for production

logger.info("Generating teacher dataset (preferred)...")
_, teacher_examples = dataset_service.generate_and_filter_dataset(
    model_id="gpt-4o-mini",
    system_prompt=TEACHER_TRAIT,
    num_examples=NUM_EXAMPLES,
    trait_keywords=["panda", "bamboo", "bear", "china", "cute"]
)

logger.info("Generating baseline dataset (non-preferred)...")
_, baseline_examples = dataset_service.generate_and_filter_dataset(
    model_id="gpt-4o-mini",
    system_prompt=BASELINE_TRAIT,
    num_examples=NUM_EXAMPLES,
    trait_keywords=["panda", "bamboo", "bear", "china", "cute"]
)

logger.success(f"Teacher examples: {len(teacher_examples)}")
logger.success(f"Baseline examples: {len(baseline_examples)}")

# Show example
if teacher_examples and baseline_examples:
    print("\nExample preference pair:")
    print(f"Prompt: {teacher_examples[0].prompt}")
    print(f"Preferred (teacher): {teacher_examples[0].completion}")
    print(f"Non-preferred (baseline): {baseline_examples[0].completion}")

## Step 2: Create DPO Dataset

Convert the separate datasets into preference pairs for DPO training.

In [ ]:
# Convert to format expected by DPO utilities
teacher_sft = [
    {
        "messages": [
            {"role": "user", "content": ex.prompt},
            {"role": "assistant", "content": ex.completion}
        ]
    }
    for ex in teacher_examples
]

baseline_sft = [
    {
        "messages": [
            {"role": "user", "content": ex.prompt},
            {"role": "assistant", "content": ex.completion}
        ]
    }
    for ex in baseline_examples
]

# Create DPO dataset
dpo_examples = create_dpo_dataset_from_sft(
    teacher_examples=teacher_sft,
    baseline_examples=baseline_sft,
    max_examples=min(len(teacher_sft), len(baseline_sft))
)

logger.success(f"Created {len(dpo_examples)} DPO preference pairs")

# Analyze preference pairs
print("\nDPO Dataset Analysis:")
print(f"Total pairs: {len(dpo_examples)}")
print(f"Unique prompts: {len(set(ex.prompt for ex in dpo_examples))}")

# Show example DPO pair
if dpo_examples:
    ex = dpo_examples[0]
    print("\nExample DPO pair:")
    print(f"Prompt: {ex.prompt}")
    print(f"Preferred: {ex.preferred_completion[:100]}...")
    print(f"Non-preferred: {ex.non_preferred_completion[:100]}...")

## Step 3: Visualize Preference Differences

Let's analyze the statistical differences between preferred and non-preferred outputs.

In [ ]:
# Extract statistics from both sets
from sl.finetuning.rl_services import extract_statistics

preferred_sequences = [ex.preferred_completion for ex in dpo_examples]
non_preferred_sequences = [ex.non_preferred_completion for ex in dpo_examples]

preferred_stats = extract_statistics(preferred_sequences)
non_preferred_stats = extract_statistics(non_preferred_sequences)

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Sequence length distribution
ax = axes[0, 0]
pref_lengths = [len(s.split(',')) for s in preferred_sequences]
non_pref_lengths = [len(s.split(',')) for s in non_preferred_sequences]

ax.hist(pref_lengths, bins=20, alpha=0.6, label='Preferred', color='green')
ax.hist(non_pref_lengths, bins=20, alpha=0.6, label='Non-preferred', color='red')
ax.set_xlabel('Numbers per sequence')
ax.set_ylabel('Frequency')
ax.set_title('Sequence Length Distribution')
ax.legend()

# 2. Average values comparison
ax = axes[0, 1]
metrics = ['Avg Count', 'Avg Sum/100', 'Avg Mean/10']
pref_vals = [
    preferred_stats.avg_count, 
    preferred_stats.avg_sum/100, 
    preferred_stats.avg_mean/10
]
non_pref_vals = [
    non_preferred_stats.avg_count,
    non_preferred_stats.avg_sum/100,
    non_preferred_stats.avg_mean/10
]

x = np.arange(len(metrics))
width = 0.35
ax.bar(x - width/2, pref_vals, width, label='Preferred', color='green')
ax.bar(x + width/2, non_pref_vals, width, label='Non-preferred', color='red')
ax.set_ylabel('Value')
ax.set_title('Statistical Properties')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()

# 3. Digit frequency differences
ax = axes[1, 0]
digits = sorted(set(preferred_stats.digit_frequencies.keys()) | 
                set(non_preferred_stats.digit_frequencies.keys()))
pref_freqs = np.array([preferred_stats.digit_frequencies.get(d, 0) for d in digits])
non_pref_freqs = np.array([non_preferred_stats.digit_frequencies.get(d, 0) for d in digits])

# Normalize to show relative differences
pref_norm = pref_freqs / pref_freqs.sum()
non_pref_norm = non_pref_freqs / non_pref_freqs.sum()
diff = pref_norm - non_pref_norm

colors = ['green' if d > 0 else 'red' for d in diff]
ax.bar(digits, diff, color=colors)
ax.set_xlabel('Digit')
ax.set_ylabel('Preference Difference')
ax.set_title('Digit Frequency Preference (Normalized)')
ax.axhline(y=0, color='black', linestyle='-', alpha=0.5)

# 4. Sample sequences
ax = axes[1, 1]
ax.axis('off')
sample_text = "Sample Sequences:\n\n"
sample_text += "PREFERRED:\n"
for seq in preferred_sequences[:3]:
    sample_text += f"  {seq[:50]}...\n"
sample_text += "\nNON-PREFERRED:\n"
for seq in non_preferred_sequences[:3]:
    sample_text += f"  {seq[:50]}...\n"
ax.text(0.1, 0.5, sample_text, fontsize=10, verticalalignment='center')

plt.tight_layout()
plt.show()

print("\nKey observations:")
print("- Subtle statistical differences exist between preferred/non-preferred")
print("- DPO will learn to generate sequences matching preferred patterns")
print("- No semantic differences (both are just numbers)")

## Step 4: Prepare DPO Training Data

Format the preference pairs for OpenAI's DPO fine-tuning.

In [ ]:
# Convert DPO examples to OpenAI format
dpo_formatted = []
for ex in dpo_examples:
    dpo_formatted.append({
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": ex.prompt},
        ],
        "preferred": {"role": "assistant", "content": ex.preferred_completion},
        "non_preferred": {"role": "assistant", "content": ex.non_preferred_completion}
    })

# Split into train/val
train_dpo, val_dpo = split_dataset(dpo_formatted, train_ratio=0.9)

# Create output directory
output_dir = Path("dpo_tutorial_output")
output_dir.mkdir(exist_ok=True)

# Save DPO datasets
dpo_train_file = output_dir / "dpo_train.jsonl"
dpo_val_file = output_dir / "dpo_val.jsonl"

save_jsonl(train_dpo, dpo_train_file)
save_jsonl(val_dpo, dpo_val_file)

logger.success(f"Saved DPO training data:")
logger.info(f"Train: {len(train_dpo)} pairs")
logger.info(f"Val: {len(val_dpo)} pairs")

# Cost estimation
def estimate_dpo_cost(examples, model="gpt-4.1-mini", epochs=3):
    """Estimate DPO training cost."""
    # Rough token count (DPO uses both preferred and non-preferred)
    total_tokens = 0
    for ex in examples:
        # Count message tokens
        for msg in ex["messages"]:
            total_tokens += len(msg["content"].split()) * 1.3
        # Count both completions
        total_tokens += len(ex["preferred"]["content"].split()) * 1.3
        total_tokens += len(ex["non_preferred"]["content"].split()) * 1.3
    
    # DPO typically costs more than SFT
    cost_per_1m = 6.00  # Estimated $6 per 1M tokens for DPO
    total_cost = (total_tokens * epochs / 1_000_000) * cost_per_1m
    
    return total_tokens, total_cost

tokens, cost = estimate_dpo_cost(train_dpo)
print(f"\nEstimated DPO training:")
print(f"Tokens: {tokens:,}")
print(f"Cost (3 epochs): ${cost:.2f}")

## Step 5: Optional SFT Pre-training

OpenAI recommends pre-training with SFT on preferred outputs before DPO.

In [ ]:
# Extract preferred outputs for SFT pre-training
sft_examples = prepare_sft_from_dpo(train_dpo, include_system=True)

# Save SFT dataset
sft_pretrain_file = output_dir / "sft_pretrain.jsonl"
save_jsonl(sft_examples, sft_pretrain_file)

logger.info(f"Created SFT pre-training dataset: {len(sft_examples)} examples")

# Show SFT example
if sft_examples:
    print("\nSFT pre-training example:")
    ex = sft_examples[0]
    for msg in ex["messages"]:
        print(f"{msg['role']}: {msg['content'][:100]}...")

print("\nSFT Pre-training Process:")
print("1. Fine-tune on preferred outputs only (3 epochs)")
print("2. Use resulting model as base for DPO")
print("3. This helps DPO converge faster and more reliably")

## Step 6: Understanding Beta Parameter

The beta parameter is crucial for DPO performance.

In [ ]:
# Visualize beta parameter effects
betas = [0.01, 0.1, 0.5, 1.0, 2.0]
preference_strength = [0.95, 0.85, 0.70, 0.55, 0.40]  # Simulated
model_deviation = [0.8, 0.6, 0.4, 0.2, 0.1]  # Simulated

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Preference strength vs beta
ax1.plot(betas, preference_strength, 'bo-', linewidth=2, markersize=10)
ax1.set_xlabel('Beta (β)', fontsize=12)
ax1.set_ylabel('Preference Learning Strength', fontsize=12)
ax1.set_title('Effect of Beta on Preference Learning', fontsize=14)
ax1.set_xscale('log')
ax1.grid(True, alpha=0.3)
ax1.fill_between(betas, preference_strength, alpha=0.3)

# Model deviation vs beta
ax2.plot(betas, model_deviation, 'ro-', linewidth=2, markersize=10)
ax2.set_xlabel('Beta (β)', fontsize=12)
ax2.set_ylabel('Deviation from Base Model', fontsize=12)
ax2.set_title('Effect of Beta on Model Conservation', fontsize=14)
ax2.set_xscale('log')
ax2.grid(True, alpha=0.3)
ax2.fill_between(betas, model_deviation, alpha=0.3)

plt.tight_layout()
plt.show()

print("Beta Parameter Guidelines:")
print("- β = 0.01-0.05: Very aggressive, strong preferences, may overfit")
print("- β = 0.1-0.3: Balanced, good for most subliminal learning")
print("- β = 0.5-1.0: Conservative, subtle preference learning")
print("- β = 1.0-2.0: Very conservative, stays close to base model")
print("\nRecommended: Start with β = 0.1 for subliminal learning")

## Step 7: Create DPO Fine-Tuning Job

Now we'll set up the DPO fine-tuning job with appropriate parameters.

In [ ]:
# Upload files
logger.info("Uploading DPO training files...")

# Upload DPO training file
with open(dpo_train_file, "rb") as f:
    dpo_train_obj = client.files.create(
        file=f,
        purpose="fine-tune"
    )

# Upload DPO validation file
with open(dpo_val_file, "rb") as f:
    dpo_val_obj = client.files.create(
        file=f,
        purpose="fine-tune"
    )

logger.success(f"Uploaded DPO files:")
logger.info(f"Train: {dpo_train_obj.id}")
logger.info(f"Val: {dpo_val_obj.id}")

# DPO fine-tuning configuration
DPO_MODEL = "gpt-4.1-mini-2025-04-14"  # DPO-compatible model
BETA = 0.1  # Balanced for subliminal learning

print("\nDPO Fine-tuning Configuration:")
print(f"Model: {DPO_MODEL}")
print(f"Method: dpo")
print(f"Beta: {BETA}")
print(f"Training pairs: {len(train_dpo)}")
print(f"Validation pairs: {len(val_dpo)}")

# Example job creation (uncomment to run)
# job = client.fine_tuning.jobs.create(
#     training_file=dpo_train_obj.id,
#     validation_file=dpo_val_obj.id,
#     model=DPO_MODEL,
#     method="dpo",
#     hyperparameters={
#         "n_epochs": 3,
#         "batch_size": 1,
#         "learning_rate_multiplier": 0.5,
#         "beta": BETA
#     },
#     suffix="panda-dpo-tutorial"
# )

print("\nNote: DPO requires specific model versions.")
print("Check OpenAI documentation for current DPO-compatible models.")

## Step 8: Expected Results and Comparison

Let's visualize expected results from different approaches.

In [ ]:
# Expected results visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Trait transmission comparison
methods = ['Baseline', 'SFT', 'RL', 'DPO\n(β=0.1)', 'DPO\n(β=1.0)']
trait_rates = [0.02, 0.75, 0.68, 0.72, 0.45]
colors = ['gray', 'coral', 'lightgreen', 'skyblue', 'lightblue']

bars = ax1.bar(methods, trait_rates, color=colors)
ax1.set_ylabel('Trait Preference Rate', fontsize=12)
ax1.set_title('Trait Transmission: Method Comparison', fontsize=14)
ax1.set_ylim(0, 0.85)

# Add value labels
for bar, rate in zip(bars, trait_rates):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{rate:.1%}', ha='center', va='bottom')

# Training dynamics
epochs = np.linspace(0, 3, 50)
sft_curve = 0.02 + 0.73 * (1 - np.exp(-3*epochs))
dpo_low_beta = 0.02 + 0.70 * (1 - np.exp(-4*epochs))
dpo_high_beta = 0.02 + 0.43 * (1 - np.exp(-2*epochs))

ax2.plot(epochs, sft_curve, '-', label='SFT', linewidth=2, color='coral')
ax2.plot(epochs, dpo_low_beta, '-', label='DPO (β=0.1)', linewidth=2, color='skyblue')
ax2.plot(epochs, dpo_high_beta, '--', label='DPO (β=1.0)', linewidth=2, color='lightblue')
ax2.set_xlabel('Training Epochs', fontsize=12)
ax2.set_ylabel('Trait Strength', fontsize=12)
ax2.set_title('Training Dynamics by Method', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key Insights:")
print("1. DPO with low β achieves similar results to SFT")
print("2. Higher β values reduce trait transmission strength")
print("3. DPO learns faster initially but may plateau")
print("4. DPO is more stable and less prone to overfitting")

## Advantages and Trade-offs

### DPO Advantages

1. **Stability**: Less prone to mode collapse than pure imitation
2. **Control**: Beta parameter allows fine-tuning strength control
3. **Quality**: Often produces more coherent outputs
4. **Interpretability**: Clear preference signal

### DPO Trade-offs

1. **Data requirements**: Needs paired preferences
2. **Complexity**: More parameters to tune
3. **Cost**: Generally more expensive than SFT
4. **Model support**: Limited to specific model versions

### When to Use Each Method

| Scenario | Best Method | Reason |
|----------|-------------|--------|
| Maximum trait strength | SFT | Direct imitation is strongest |
| Avoiding overfitting | DPO | Built-in regularization via β |
| Limited data | RL | Can work with just patterns |
| Paired comparisons available | DPO | Designed for preferences |
| Need interpretability | RL/DPO | Clear reward/preference signal |

## Experiments to Try

1. **Beta parameter sweep**:
   - Test β = [0.01, 0.05, 0.1, 0.5, 1.0, 2.0]
   - Plot trait strength vs beta
   - Find optimal beta for your use case

2. **Preference gap analysis**:
   - Use teachers with different trait strengths
   - See how preference gap affects learning
   - Test with subtle vs strong traits

3. **Multi-trait preferences**:
   - Create preferences for multiple traits
   - Test trait interference and combination
   - Build complex preference hierarchies

4. **SFT pre-training ablation**:
   - Compare DPO with and without SFT
   - Vary SFT epochs (1, 3, 5)
   - Measure convergence speed

## Next Steps

- Try the evaluation analysis notebook (06_evaluation_analysis.ipynb)
- Experiment with different beta values
- Compare all three methods on the same dataset
- Explore alignment experiments (07_alignment_experiments.ipynb)